In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
bronze_products_table = f"{catalog}.{bronze_schema}.{data_source}"
bronze_products_table

In [0]:
df = spark.sql(f"select * from {bronze_products_table}")

df.dropDuplicates(subset = ['product_id'])

In [0]:

df = df.withColumn(
    "variant",
    regexp_extract(col("product_name"), r"\((.*?)\)", 1)
)

df = df.withColumn(
    "product_name",
    regexp_replace(col("product_name"), r"\s*\(.*?\)", "")
)

df = df.withColumn(
    "category",
    initcap(col("category"))
)

In [0]:
df.columns

In [0]:
display(df)

In [0]:
df = df.withColumn(
    "product_name",
    when(
        col("product_name") == "SportsBar Protien Bar Peanut Crunch","SportsBar Protein Bar Peanut Crunch"
    ).otherwise(col("product_name"))
)
df = df.withColumn(
    "category",
    when(
        col("category") == "Protien Bars","Protein Bars"
    ).otherwise(col("category"))
)

In [0]:
display(df)

In [0]:
display(df.select("category").distinct())

In [0]:
df = df.withColumn(
    "division",
    when(col("category") == "Protein Bars",         "Nutrition Bars")
    .when(col("category") == "Energy Bars",         "Nutrition Bars")
    .when(col("category") == "Granola & Cereals" ,  "Breakfast Foods")
    .when(col("category") == "Recovery Dairy",       "Dairy & Recovery")
    .when(col("category") == "Healthy Snacks",      "Healthy Snacks")
    .when(col("category") == "Electrolyte Mix",      "Hydration& Electrolytes") 
    .otherwise("Other")

)

In [0]:
INVALID_PRODUCT_ID = "99999"
df = df.withColumn(
    "product_id",
    when(
        col("product_id").cast("string").rlike("^[0-9]+$"),
        col("product_id").cast("string")
    ).otherwise(INVALID_PRODUCT_ID)
).withColumn(
    "product_code",
    sha2(col("product_id").cast("string"), 256)
)



In [0]:
df.columns

In [0]:
df = df.select(
 'product_code',
 'product_id',
 'product_name',
 'category',
 'variant',
 'division',
 'ingest_timestamp',
 'file_name'
 )
display(df)

In [0]:
silver_product_table = f"{catalog}.{silver_schema}.{data_source}"
df.write.format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema","true")\
    .saveAsTable(silver_product_table)